# 01 — Data Loading

This notebook downloads and stores the public market data used for the
five-day volatility prediction project.

The data will be downloaded from Yahoo Finance using `yfinance` and saved
to the `data/raw/` directory for use in subsequent notebooks.

This notebook has following objectives:

- Download historical SPY market data.
- Inspect the downloaded dataset.
- Perform basic checks on the data structure and date range.
- Save the raw data in a reproducible format.

The target variable and derived features will be constructed in later
notebooks.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

from pathlib import Path

get their versions:

In [2]:
print(f"NumPy version:   {np.__version__}")
print(f"Pandas version:  {pd.__version__}")
print(f"yfinance version: {yf.__version__}")

NumPy version:   2.4.6
Pandas version:  3.0.3
yfinance version: 1.4.0


## 2. Define Project Paths

In [3]:
# Project directories

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")

Project root: /Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility
Raw data directory: /Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility/data/raw


## 3. Get The Data
### 3.1: Choice of Historical Period

We use daily SPY data from January 2010 to the present.

The period is intentionally long enough to include several qualitatively
different market and volatility regimes, including the relatively low-volatility
period of the mid-2010s, major volatility shocks such as the COVID-19 crisis,
the inflation and monetary-tightening period beginning in 2022, and the more
recent market environment.

Although these regimes are not identical, this variation is useful for studying
whether volatility dynamics and model predictions remain informative across
different market conditions. It is also particularly relevant for the
topological and dynamical-systems analyses developed later in the project.

To avoid information leakage, model evaluation will use chronological
train/validation/test splits rather than random splits. Thus, models will be
trained on earlier observations and evaluated on later periods.

The data loader will download observations through the current date rather than
using a fixed end date, keeping the project reproducible and usable in the
future.

### 3.2: SPY Market Data

We use SPY as a liquid, broad-market proxy for the U.S. equity market.

For each trading day, we download daily Open, High, Low, Close, and Volume
data. Prices are adjusted for corporate actions so that returns can be
computed consistently across the full historical period.

The data are downloaded using `yfinance`, with January 1, 2010 as the starting
date and the current date as the end of the sample.

The raw dataset will be stored locally in `data/raw/`. The raw data will not
be committed to GitHub; the download procedure in this notebook makes the
dataset reproducible.

In [4]:
# Download daily SPY market data

TICKER = "SPY"
START_DATE = "2010-01-01"

spy_data = yf.download(
    TICKER,
    start=START_DATE,
    end=pd.Timestamp.today().strftime("%Y-%m-%d"),
    interval="1d",
    auto_adjust=True,
    progress=False
)

spy_data.head()

Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
2010-01-04,84.578491,84.623268,83.220220,83.862041,118944600
2010-01-05,84.802376,84.839693,84.220261,84.526247,111579900
2010-01-06,84.862091,85.071055,84.653127,84.720292,116074400
2010-01-07,85.220306,85.324788,84.466539,84.705356,131091100
2010-01-08,85.503906,85.541224,84.824774,84.996420,126402800


### 3.3: Simplify the Column Structure

`yfinance` returns the downloaded data with a hierarchical column index,
even when only a single ticker is requested.

Since this project uses only SPY, we flatten the columns to obtain a
standard tabular structure with the columns `Open`, `High`, `Low`, `Close`,
and `Volume`.

In [5]:
# Flatten the column index

spy_data.columns = spy_data.columns.get_level_values(0)

spy_data.head()

Price,Close,High,Low,Open,Volume
2010-01-04,84.578491,84.623268,83.220220,83.862041,118944600
2010-01-05,84.802376,84.839693,84.220261,84.526247,111579900
2010-01-06,84.862091,85.071055,84.653127,84.720292,116074400
2010-01-07,85.220306,85.324788,84.466539,84.705356,131091100
2010-01-08,85.503906,85.541224,84.824774,84.996420,126402800


### 3.4: Basic Data Checks

Before saving the raw dataset, we perform a few basic checks on its
structure, date range, and missing values.

These checks help ensure that the downloaded data are suitable for
subsequent analysis.

In [6]:
# Inspect the shape of the dataset

print(f"Number of observations: {len(spy_data):,}")
print(f"Number of columns: {spy_data.shape[1]}")
print(f"First date: {spy_data.index.min().date()}")
print(f"Last date: {spy_data.index.max().date()}")

Number of observations: 4,191
Number of columns: 5
First date: 2010-01-04
Last date: 2026-09-01


check for missing values:

In [7]:
spy_data.isna().sum()

Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

and data type:

In [8]:
spy_data.dtypes

Price
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

### 3.5: Save the Raw Data

The downloaded SPY data are saved in CSV format in the `data/raw/`
directory.

The raw data are excluded from version control through `.gitignore`.
Therefore, the notebook records the procedure used to reproduce the
dataset without committing the downloaded market data to GitHub.

In [9]:
# Save the raw data

RAW_DATA_FILE = RAW_DATA_DIR / "spy_daily.csv"

spy_data.to_csv(RAW_DATA_FILE)

print(f"Raw data saved to: {RAW_DATA_FILE}")

Raw data saved to: /Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility/data/raw/spy_daily.csv


verify that the file has been written correctly:

In [10]:
# Verify the saved file

print(f"File exists: {RAW_DATA_FILE.exists()}")
print(f"File size: {RAW_DATA_FILE.stat().st_size / 1024:.1f} KB")

File exists: True
File size: 383.8 KB


### 3.6: Final Dataset Preview

The raw dataset is now stored locally and will serve as the input for
the subsequent notebooks.

No returns, volatility measures, target variables, or predictive features
are constructed in this notebook. These transformations are deliberately
left to later stages of the project.

In [11]:
# Display the final dataset preview

spy_data.tail()

Price,Close,High,Low,Open,Volume
2026-08-26,766.080017,767.349976,763.929993,764.729980,28751600
2026-08-27,771.099976,772.359985,767.159973,768.500000,34557100
2026-08-28,769.349976,775.299988,768.309998,771.760010,36744300
2026-08-31,767.049988,768.000000,764.719971,767.330017,38810800
2026-09-01,761.780029,764.669983,759.479980,762.010010,41054800
